# Step 5b: accuracy fallback for the monsoon (2016–2017)

The analysis plan (Section 3) says: if an offshore accuracy unit has fewer than 50 matchups in 2018–2020, its window is extended back to 2016–2017. Five monsoon units are short (all three products by day; Aqua and Terra by night). This notebook downloads only June–September of 2016 and 2017: the iQuam drifter files, the three L3 products, and runs Step 5 on them. Nothing from 2018–2020 is changed.

You need: `nbob_sst_study_v2026-10-13b.zip`, your NASA Earthdata login, your Copernicus Marine login, and the `nbob_step5` folder from Step 5 in your Drive. Expect about 1–2 hours, mostly downloading. Every download cell can be run again after a disconnect; finished files are skipped.

Run the cells from top to bottom (Shift+Enter).

## 1. Clean start and install

In [ ]:
%cd /content
!rm -rf /content/nbob_sst_study /content/*.zip
!pip -q install xarray netCDF4 h5netcdf siphon earthaccess copernicusmarine pandas requests matplotlib

## 2. Upload `nbob_sst_study_v2026-10-13b.zip`

In [ ]:
from google.colab import files
uploaded = files.upload()
zipname = list(uploaded.keys())[0]
!unzip -o -q "$zipname" -d /content
%cd /content/nbob_sst_study
!cat VERSION

**Check:** the cell above must print `2026-10-13b` (or later).

## 3. Connect Drive and check the Step 5 folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive'
IQX = f'{D}/nbob_step2_iquam_ext'
L3X = f'{D}/nbob_step4_l3_ext'
OUT5 = f'{D}/nbob_step5'
OUT5X = f'{D}/nbob_step5_ext'
for p in (IQX, L3X, OUT5X):
    os.makedirs(p, exist_ok=True)
for f in ['matchups.csv', 'bathymetry_nbob.nc']:
    print(f, 'found' if os.path.exists(f'{OUT5}/{f}') else 'MISSING: check the Step 5 folder name')

**Check:** both lines must say `found`.

## 4. iQuam drifters, June–September 2016 and 2017 (about 10–20 minutes)
Each line is one month: records in the box, drifter records, seconds.

In [ ]:
!python iquam_scope_auto.py --start 2016-06 --end 2016-09 --out "$IQX" 2>&1 | tee -a "$IQX/log.txt"
!python iquam_scope_auto.py --start 2017-06 --end 2017-09 --out "$IQX" 2>&1 | tee -a "$IQX/log.txt"

## 5. NASA Earthdata login (for MetOp-B)

In [ ]:
import getpass, earthaccess
user = input('Earthdata username: ')
pw = getpass.getpass('Earthdata password: ')
netrc = os.path.expanduser('~/.netrc')
with open(netrc, 'w') as f:
    f.write(f'machine urs.earthdata.nasa.gov login {user} password {pw}\n')
os.chmod(netrc, 0o600)
auth = earthaccess.login(strategy='netrc')
print('Login OK' if auth.authenticated else 'Login FAILED: check username and password')

## 6. L3 downloads, monsoon months only
Three cells, one per product. The MODIS cells first read the NOAA catalogue, which can take 10–20 minutes before downloads start. If MetOp-B reports 0 files, that is acceptable: continue.

In [ ]:
!python step4_l3_download.py --product metopb --start 2016-06-01 --end 2017-09-30 --months 6 7 8 9 --out "$L3X" --workers 4 2>&1 | tee -a "$L3X/log_metopb.txt"

In [ ]:
!python step4_l3_download.py --product aqua --start 2016-06-01 --end 2017-09-30 --months 6 7 8 9 --out "$L3X" --workers 4 2>&1 | tee -a "$L3X/log_aqua.txt"

In [ ]:
!python step4_l3_download.py --product terra --start 2016-06-01 --end 2017-09-30 --months 6 7 8 9 --out "$L3X" --workers 4 2>&1 | tee -a "$L3X/log_terra.txt"

In [ ]:
!python step4_l3_download.py --inventory --out "$L3X" 2>&1 | tee "$L3X/log_inventory.txt"

**Check:** about 488 files for each MODIS product (244 days, day and night files). If a cell shows `FAILED` lines, run that cell again.

## 7. Copernicus Marine login (daily L4 for June 2016 to September 2017)

In [ ]:
import copernicusmarine
user = input('Copernicus Marine username: ')
pw = getpass.getpass('Password: ')
copernicusmarine.login(username=user, password=pw)

## 8. Step 5 on the 2016–2017 monsoon data (about 10 minutes)
The same code as before, with the same bathymetry file as the 2018–2020 run. Then the quality-level-5 check.

In [ ]:
!python step5_reliability.py --l3 "$L3X" --iquam "$IQX/box_monthly" --bathy "$OUT5/bathymetry_nbob.nc" --start 2016-06-01 --end 2017-09-30 --out "$OUT5X" 2>&1 | grep -v Warning | grep -v 'np.nanmean' | tee "$OUT5X/log_ext.txt"
!python step5_reliability.py --l3 "$L3X" --iquam "$IQX/box_monthly" --bathy "$OUT5/bathymetry_nbob.nc" --l4 "$OUT5X/l4_daily_stationarea.nc" --start 2016-06-01 --end 2017-09-30 --ql-min 5 --out "$OUT5X/check_ql5" 2>&1 | grep -v Warning | grep -v 'np.nanmean' | tail -40

## 9. Download the results
Send `step5b_results.zip` back in the chat.

In [ ]:
!cd "$D" && zip -q -r /content/step5b_results.zip nbob_step5_ext/*.csv nbob_step5_ext/*.txt nbob_step5_ext/check_ql5/*.csv nbob_step5_ext/check_ql5/*.txt nbob_step2_iquam_ext/tables nbob_step2_iquam_ext/log.txt nbob_step4_l3_ext/inventory_*.csv nbob_step4_l3_ext/log_*.txt nbob_step4_l3_ext/failed_*.csv
files.download('/content/step5b_results.zip')

## If something fails
- Colab disconnected: run cells 1–3 and 5 again, then the cell that stopped. Finished files and months are skipped.
- A red error: copy the last 20 lines and send them in the chat.